In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import glob

In [2]:
def process_files_to_chunks(file_paths, output_dir, label, chunk_size=10000):
    '''
    Processes the Jet Images data from the .h5 format for given jet class
    [label] and saves the images for that class as .npy files with [chunk_size]
    images each in the directory given by [output_dir] (the last file saved
    can have less than 10,000 images).

    Args:
        - file_paths : list of str paths to the Jet Images .h5 files
        - output_dir : the directory to save the .npy files
        - label : the class label to process. Must be one of 'j_q', 'j_g', 'j_w', 'j_t', or 'j_z'
        - chunk_size : the number of images to include in each .npy file, 10,000 by default
    '''
    os.makedirs(output_dir, exist_ok=True)
    buffer = []
    file_count = 0
    
    save_label = label[-1]

    for file_path in file_paths:
        print("processing file: ", file_path)
        with h5py.File(file_path,'r') as f: 
            images = f['jetImage'][:]
            tmpDf = pd.DataFrame(list(f['jets'][:,-6:]), columns=[i.decode("utf-8") for i in list(f['jetFeatureNames'][-6:])])
            condition = tmpDf[label] == 1        
        images_i = images[condition]
        print("number satisfying condition: ", images_i.shape[0])
        buffer.extend(images_i)
          
        while len(buffer) >= chunk_size:
            chunk = np.array(buffer[:chunk_size])
            out_path = os.path.join(output_dir,"jetImages_" + save_label +"_" + str(file_count) + ".npy")
            print("saving file:", out_path)
            np.save(out_path, chunk)
            file_count += 1
            buffer = buffer[chunk_size:]

    # Save remaining if any
    if buffer:
        out_path = os.path.join(output_dir, f"jetImages_" + save_label + "_" + str(file_count) + ".npy")
        print("remaining", len(buffer))
        print("saving remaining:", out_path)
        np.save(out_path, np.array(buffer))
    return


In [8]:
# Folder ../../original_data/train/ contains the unzipped "hls4ml_LHCjet_30p_train.tar.gz" file from https://zenodo.org/records/3601436
files = glob.glob("../../original_data/train/*")
train_files

['../../original_data/train/jetImage_0_30p_0_10000.h5',
 '../../original_data/train/jetImage_0_30p_10000_20000.h5',
 '../../original_data/train/jetImage_0_30p_20000_30000.h5',
 '../../original_data/train/jetImage_0_30p_30000_40000.h5',
 '../../original_data/train/jetImage_0_30p_40000_50000.h5',
 '../../original_data/train/jetImage_0_30p_50000_60000.h5',
 '../../original_data/train/jetImage_0_30p_60000_70000.h5',
 '../../original_data/train/jetImage_0_30p_70000_80000.h5',
 '../../original_data/train/jetImage_0_30p_80000_90000.h5',
 '../../original_data/train/jetImage_1_30p_0_10000.h5',
 '../../original_data/train/jetImage_1_30p_10000_20000.h5',
 '../../original_data/train/jetImage_1_30p_20000_30000.h5',
 '../../original_data/train/jetImage_1_30p_30000_40000.h5',
 '../../original_data/train/jetImage_1_30p_40000_50000.h5',
 '../../original_data/train/jetImage_1_30p_50000_60000.h5',
 '../../original_data/train/jetImage_1_30p_60000_70000.h5',
 '../../original_data/train/jetImage_1_30p_70000

In [ ]:
for signal in ["j_g","j_q","j_w","j_z","j_t"]:
    process_files_to_chunks(train_files,"jet_data/",signal)

In [ ]:
def crop_images(files,crop_size):
    '''
    Crops the .npy files in the [files] list to the indices given in [crop_size].
    Saves the files with the same name as the original file with a suffix of
    '_cropped'

    Args:
        - files: list of strings that give the path to the .npy files to process
        - crop_size: tuple of integers (i,j) such that the .npy images of shape [N,H,W]
                     are cropped to images[:,i:j,i:j]
                     
    '''
    min = crop_size[0]
    max = crop_size[1]
    size = max - min
    for file in files:
        print("processing: ", file)
        name = file[:file.find(".")]
        imgs = np.load(file)
        imgs_cropped = imgs[:,min:max,min:max].reshape(-1,size,size,1)
        np.save(name + "_cropped.npy",imgs_cropped)
    return

In [ ]:
g_files = glob.glob("jet_data/train/*_g_*.npy")
q_files = glob.glob("jet_data/train/*_q_*.npy")
t_files = glob.glob("jet_data/train/*_t_*.npy")
w_files = glob.glob("jet_data/train/*_w_*.npy")
z_files = glob.glob("jet_data/train/*_z_*.npy")

crop_size = (38,62)
for files_set in [g_files_train,
                  q_files_train,
                  t_files_train,
                  w_files_train,
                  z_files_train]:
    crop_images(files_set)

Then organize the cropped files in the following structure:

    jet_data/
        train/
            jetImages_g_02_cropped.npy
            jetImages_g_03_cropped.npy
            ...
            jetImages_q_02_cropped.npy
            jetImages_q_03_cropped.npy
            ...
        test/
            jetImages_g_00_cropped.npy
            jetImages_q_00_cropped.npy
        val/
            jetImages_g_01_cropped.npy
            jetImages_q_01_cropped.npy
        signals/
            jetImages_w_00_cropped.npy
            ...
            jetImages_z_00_cropped.npy
            ...
            jetImages_t_00_cropped.npy
            ...

Note that for evaluation of the modules, only the first 4 files of each w,z,t data were utilized for a total of 40k signal events per class.